In [32]:
import pandas as pd

import numpy as np

df = pd.read_csv('../data/raw/blinkit_dataset (2).csv')


print(f"Raw shape: {df.shape}")


Raw shape: (13000, 25)


##  1: Fix date columns
- Converting date_added and expiry_date from string to datetime

In [33]:
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')

df['expiry_date'] = pd.to_datetime(df['expiry_date'], errors='coerce')

print("Transformation 1: date_added and expiry_date converted to datetime")

print(f"  date_added nulls after conversion: {df['date_added'].isnull().sum()}")

print(f"  expiry_date nulls after conversion: {df['expiry_date'].isnull().sum()}")


Transformation 1: date_added and expiry_date converted to datetime
  date_added nulls after conversion: 0
  expiry_date nulls after conversion: 0


## 2: Standardised text columns
- Strip whitespace and enforce title case on categorical columns


In [34]:
text_cols = ['category', 'brand', 'city', 'seller', 'packaging_type', 'delivery_status', 'offer_type']

for col in text_cols:
    df[col] = df[col].str.strip().str.title()

print("Transformation 4: Text columns standardised (strip + title case)")


Transformation 4: Text columns standardised (strip + title case)


## 3: Validate price logic
- Flag rows where final_price > price (should never happen after discount)


In [35]:
df['price_logic_check'] = df['final_price'] <= df['price']

invalid_price = df[~df['price_logic_check']].shape[0]

print(f"Transformation 5: Price logic check — {invalid_price} anomalies found")

# Drop the check column

df.drop('price_logic_check', axis=1, inplace=True)


Transformation 5: Price logic check — 0 anomalies found


4: Create revenue and revenue_per_unit columns
- Business KPI columns derived for analysis

In [36]:
df['revenue'] = df['final_price'] * df['sold_quantity']

df['revenue_per_unit'] = df['final_price']

df['profit'] = df['revenue'] * (df['profit_margin_pct'] / 100)

print("Transformation 6: revenue, revenue_per_unit, profit columns created")


Transformation 6: revenue, revenue_per_unit, profit columns created


In [37]:
df

,product_id,product_name,category,brand,price,discount_pct,final_price,rating,num_reviews,delivery_time_min,...,shelf_life_days,reorder_level,demand_index,date_added,expiry_date,offer_type,delivery_status,revenue,revenue_per_unit,profit
0,1,Tata Organic Grocery 300,Grocery,Tata,199.78,25,149.84,4.5,146,37,...,212,15,73,2023-11-27,2024-06-26,NaN,On-Time,36111.44,149.84,10761.20912
1,2,Mother Dairy Lite Dairy 275,Dairy,Mother Dairy,44.32,30,31.02,4.0,264,36,...,17,24,25,2024-08-07,2024-08-24,NaN,Delayed,868.56,31.02,132.02112
2,3,P&G Classic Personal 439,Personal Care,P&G,501.13,0,501.13,3.7,69,17,...,1463,25,100,2024-03-03,2028-03-05,Freedelivery,On-Time,292158.79,501.13,19282.48014
3,4,Dettol Fresh Household 771,Household,Dettol,627.17,0,627.17,3.9,103,23,...,1143,18,15,2024-08-07,2027-09-24,NaN,On-Time,20696.61,627.17,7616.35248
4,5,Minute Maid Daily Beverages 264,Beverages,Minute Maid,101.69,15,86.44,4.3,422,10,...,363,30,6,2024-07-04,2025-07-02,NaN,On-Time,4149.12,86.44,601.62240
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12995,12996,Minute Maid Classic Beverages 396,Beverages,Minute Maid,219.17,15,186.29,4.8,450,30,...,482,28,73,2025-01-16,2026-05-13,NaN,On-Time,37630.58,186.29,3123.33814
12996,12997,FreshFarm Organic Fruits 225,Fruits & Vegetables,Freshfarm,348.21,5,330.80,4.5,115,25,...,13,19,35,2024-01-11,2024-01-24,Cashback,On-Time,20840.40,330.80,2188.24200
12997,12998,FreshFarm Original Fruits 349,Fruits & Vegetables,Freshfarm,178.86,10,160.97,4.8,199,19,...,3,20,34,2025-08-08,2025-08-11,NaN,On-Time,18833.49,160.97,1789.18155
12998,12999,Lizol Lite Household 792,Household,Lizol,435.70,5,413.91,3.4,114,18,...,403,24,17,2023-11-11,2024-12-18,NaN,On-Time,18212.04,413.91,6993.42336


##  5: Created is_delayed binary column


In [38]:
df['is_delayed'] = (df['delivery_status'].str.lower() == 'delayed').astype(int)

print("Transformation 7: is_delayed binary column created")

print(df['is_delayed'].value_counts())



Transformation 7: is_delayed binary column created
is_delayed
0    10395
1     2605
Name: count, dtype: int64


##  6: Add year and month from date_added

In [39]:
df['year_added'] = df['date_added'].dt.year

df['month_added'] = df['date_added'].dt.month

print("Transformation 8: year_added and month_added extracted")

Transformation 8: year_added and month_added extracted


7: Remove duplicate product_id if any

In [40]:
before = len(df)

df.drop_duplicates(subset=['product_id'], inplace=True)

after = len(df)

print(f"Transformation 9: Duplicate removal — {before - after} duplicates dropped")
if before - after > 0:
    print(f"  New shape after duplicate removal: {df.shape}")
else:
    print("  No duplicates found based on product_id")

print(f"Final shape: {df.shape}")


Transformation 9: Duplicate removal — 0 duplicates dropped
  No duplicates found based on product_id
Final shape: (13000, 31)


# Final validation


In [41]:
print("------ CLEANED DATASET SUMMARY  ------")

print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")

print(f"\nMissing values remaining:")

print(df.isnull().sum()[df.isnull().sum() > 0])

print("\nDataset is clean and ready for analysis")


------ CLEANED DATASET SUMMARY  ------
Rows: 13000 | Columns: 31

Missing values remaining:
offer_type    6544
dtype: int64

Dataset is clean and ready for analysis


## Saving cleaned file


In [42]:
df.to_csv('blinkit_cleaned.csv', index=False)

print("Saved: blinkit_cleaned.csv → upload to data/processed/ in GitHub")


Saved: blinkit_cleaned.csv → upload to data/processed/ in GitHub


In [43]:
df.columns

Index(['product_id', 'product_name', 'category', 'brand', 'price',
       'discount_pct', 'final_price', 'rating', 'num_reviews',
       'delivery_time_min', 'city', 'seller', 'stock', 'sold_quantity',
       'profit_margin_pct', 'is_organic', 'packaging_type', 'weight_g',
       'shelf_life_days', 'reorder_level', 'demand_index', 'date_added',
       'expiry_date', 'offer_type', 'delivery_status', 'revenue',
       'revenue_per_unit', 'profit', 'is_delayed', 'year_added',
       'month_added'],
      dtype='str')